# Информация о датасете
1. Кайдзен - ID предложения

2. *Подача* - Дата подачи предложения

3. *Статус* - статус предложения (обработка/согласовано/внедрено)

4. *Вид оплаты* - вид (благотворительность/1000р/и т.д.)

5. *Предложение* - текстовое сообщение предложение (имеем/предлагаем/получим)

6. *Подразд.* - подраздел

7. *Иниц.* - инициатор предложения

8. *Испол.* - исполнитель предложения


*Целевая переменная* - категория сообщения - тип к которому оно относится
Вторая предсказываемая переменная - исполнитель предложения

*Дозаполнение* таблицы данными: Подраздел по отправителю, Статус.

### Немного о блокноте

Здесь представлен достаточно черновой вариант работы, основное внимание уделялось очистке и исследованию датасета.

Работа делалась в рамках Хакатона от ПСС.

Полученная точность на модели BERT ~ 45-48%

# 1. Загрузка данных

In [5]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


### Импорт библиотек

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

from sklearn.preprocessing import MultiLabelBinarizer

### Анализ и загрузка датасета

In [7]:
df = pd.read_excel("/content/drive/MyDrive/Hakaton/data.xlsx")
df.head()

,Кайдзен+H9DA1:H1072,Подача,Статус,Вид оплаты,Предложение,Подразд.,Иниц.,Испол.
0,5236,10.01.2023,Внедрено,благотворительность,Имеем:\nВ заказах покупателя не указан номер л...,NaN,Шайдуллин Р. Ф.,Окулов А. М.
1,5239,12.01.2023,Согласование,благотворительность,Имеем:\nВ оповещении об окончании срока действ...,радуга,Кайтаева В. В.,Окулов А. М.
2,5240,12.01.2023,Внедрено,благотворительность,Имеем:\nпри расчете НДС по бизнес-единицам: АО...,NaN,Хмелева Н. В.,Окулов А. М.; Титова М. Г.
3,5242,17.01.2023,Внедрено,благотворительность,Имеем:\nИмеем базу данных кайдзен предложений....,NaN,Лучников В. А.,Окулов А. М.
4,5249,18.01.2023,Отказ,благотворительность,Имеем:\nОтуствие уведомление о поступлении на ...,офис,Чумаков А. В.,Окулов А. М.


In [8]:
# Вывод колонок и количества строк
print("Столбцы:\n", df.columns)
print("Строки (индексы):\n", df.index)

Столбцы:
 Index(['Кайдзен+H9DA1:H1072', 'Подача', 'Статус', 'Вид оплаты', 'Предложение',
       'Подразд.', 'Иниц.', 'Испол.'],
      dtype='object')
Строки (индексы):
 RangeIndex(start=0, stop=1127, step=1)


In [9]:
# Краткая сводная информация о DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1127 entries, 0 to 1126
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Кайдзен+H9DA1:H1072  1127 non-null   int64 
 1   Подача               1127 non-null   object
 2   Статус               1127 non-null   object
 3   Вид оплаты           1127 non-null   object
 4   Предложение          1127 non-null   object
 5   Подразд.             791 non-null    object
 6   Иниц.                1126 non-null   object
 7   Испол.               1127 non-null   object
dtypes: int64(1), object(7)
memory usage: 70.6+ KB


In [10]:
# Вывод количества нулевых значений в каждом столбце
df.isnull().sum()

,0
Кайдзен+H9DA1:H1072,0
Подача,0
Статус,0
Вид оплаты,0
Предложение,0
Подразд.,336
Иниц.,1
Испол.,0


In [11]:
# Уникальные значения столбцов
df[['Кайдзен+H9DA1:H1072', 'Подача', 'Статус', 'Вид оплаты', 'Подразд.', 'Иниц.', 'Испол.']].nunique()

,0
Кайдзен+H9DA1:H1072,1127
Подача,511
Статус,5
Вид оплаты,12
Подразд.,31
Иниц.,349
Испол.,279


In [12]:
# Преобразуем дату. что бы увидеть возможные закономерности
df['Подача'] = pd.to_datetime(df['Подача'], dayfirst=True, errors='coerce')

df['month'] = df['Подача'].dt.month
df['dayofweek'] = df['Подача'].dt.dayofweek

In [13]:
# приводим к нижнему регистру категориальные данные
cols = ['Предложение', 'Иниц.', 'Испол.']

for col in cols:
    df[col] = df[col].astype(str).str.lower()

In [14]:
# Преобразовываем предложение и разбиваем на 3 колонки по смыслу
def extract_parts(text):
    text = str(text).lower()

    have = re.search(r'имеем\s*[:\-]?\s*(.*?)(предлагаем|получим|$)', text, re.DOTALL)
    propose = re.search(r'предлагаем\s*[:\-]?\s*(.*?)(имеем|получим|$)', text, re.DOTALL)
    benefit = re.search(r'получим\s*[:\-]?\s*(.*)', text, re.DOTALL)

    return (
        have.group(1).strip() if have else '',
        propose.group(1).strip() if propose else '',
        benefit.group(1).strip() if benefit else ''
    )

df[['have', 'propose', 'benefit']] = df['Предложение'].apply(
    lambda x: pd.Series(extract_parts(x))
)
df

,Кайдзен+H9DA1:H1072,Подача,Статус,Вид оплаты,Предложение,Подразд.,Иниц.,Испол.,month,dayofweek,have,propose,benefit
0,5236,2023-01-10,Внедрено,благотворительность,имеем:\nв заказах покупателя не указан номер л...,NaN,шайдуллин р. ф.,окулов а. м.,1,1,в заказах покупателя не указан номер лота.,указывать номер лота в заказе покупателя.,"сокращение потери времени, на поиск номера лот..."
1,5239,2023-01-12,Согласование,благотворительность,имеем:\nв оповещении об окончании срока действ...,радуга,кайтаева в. в.,окулов а. м.,1,3,в оповещении об окончании срока действия серти...,добавить в оповещение информацию о реестре и о...,сокращение времени на выяснение какую сертифик...
2,5240,2023-01-12,Внедрено,благотворительность,имеем:\nпри расчете ндс по бизнес-единицам: ао...,NaN,хмелева н. в.,окулов а. м.; титова м. г.,1,3,при расчете ндс по бизнес-единицам: ао ппмтс п...,"в документе: ""псс:заявка на отгрузку"" реализов...",1.автоматизация учета\n2.точный расчет ндс\n3....
3,5242,2023-01-17,Внедрено,благотворительность,имеем:\nимеем базу данных кайдзен предложений....,NaN,лучников в. а.,окулов а. м.,1,1,имеем базу данных кайдзен предложений.\nзатруд...,возможность фильтрации не по одному инициатору...,"получим ускоренную обратную связь, проверить с..."
4,5249,2023-01-18,Отказ,благотворительность,имеем:\nотуствие уведомление о поступлении на ...,офис,чумаков а. в.,окулов а. м.,1,2,отуствие уведомление о поступлении на склад ож...,сделать возможность получения уведомдения на п...,оперативное отслеживание поступления важных и ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1122,8383,2026-03-12,Согласование,1000,"имеем:\nдля покраски коффердама грунтом, некот...",NaN,авдаев м. м.; гагарин д. н.,пьянков о. н.,3,3,"для покраски коффердама грунтом, некоторые час...",металлические пластины для закрытия таких мест,процесс подготовки ускорится в несколько раз
1123,8384,2026-03-12,Согласование,благотворительность,"имеем:\nдля покраски коффердама грунтом, некот...",NaN,авдаев м. м.; гагарин д. н.,пьянков о. н.,3,3,"для покраски коффердама грунтом, некоторые час...",вырезать кольца из магнитного листа,процесс подготовки ускорится в несколько раз
1124,8385,2026-03-12,Согласование,1000,"имеем:\nмаляру, который пришел на другой участ...",NaN,авдаев м. м.,пьянков о. н.; францкевич а. р.,3,3,"маляру, который пришел на другой участок, прих...",создать базу в электронном виде. где будет под...,другие маляры не будут отрываться от своей раб...
1125,8386,2026-03-12,Согласование,1000,"имеем:\nпри покраске единичных, маленьких дета...",NaN,авдаев м. м.,пьянков о. н.,3,3,"при покраске единичных, маленьких деталей. при...",приобрести малогабаритную камеру/печь.,быстрый процесс покраски мелких деталей. не пр...


In [15]:
# # Делаем списки для инициаторов и исполнителей
# def split_people(x):
#     if pd.isna(x):
#         return []
#     return [i.strip().lower() for i in re.split(r'[;,]', x)]

def split_people(x):
    """Разделяет строку на список людей, чистит пустые и мусорные значения, приводит к lower case"""
    if pd.isna(x):
        return []
    cleaned = []
    for name in re.split(r'[;,]', x):
        name = name.strip().lower()
        if not name:
            continue  # убираем пустые
        if re.match(r'^[\.\s]+$', name):
            continue
        cleaned.append(name)
    return cleaned

df['initiators'] = df['Иниц.'].apply(split_people)
df['executors'] = df['Испол.'].apply(split_people)

In [16]:
# исполнители — target
mlb_exec = MultiLabelBinarizer()
y_exec = mlb_exec.fit_transform(df['executors'])

# инициаторы — признаковый столбец
mlb_init = MultiLabelBinarizer()
X_init = mlb_init.fit_transform(df['initiators'])

In [17]:
# считаем частоту
exec_counts = Counter([i for sublist in df['executors'] for i in sublist])

# порог
thresh = 7

# функция для замены редких
def replace_rare(lst, counts, thresh=5):
    return [i if counts[i] >= thresh else 'other' for i in lst]

df['executors'] = df['executors'].apply(lambda lst: replace_rare(lst, exec_counts, thresh))

# 2. Очистка

In [18]:
# Удаляем ID
df.drop(['Кайдзен+H9DA1:H1072'], axis=1, inplace=True)
# df.drop(['Подача'], axis=1, inplace=True)

In [19]:
# Удаляем  Статус и Вид Оплаты так как он подаваться в модель не будет
df.drop(['Статус'], axis=1, inplace=True)
df.drop(['Вид оплаты'], axis=1, inplace=True)

In [20]:
df.drop(['Предложение'], axis=1, inplace=True)
df.drop(['Иниц.'], axis=1, inplace=True)
df.drop(['Испол.'], axis=1, inplace=True)
df

,Подача,Подразд.,month,dayofweek,have,propose,benefit,initiators,executors
0,2023-01-10,NaN,1,1,в заказах покупателя не указан номер лота.,указывать номер лота в заказе покупателя.,"сокращение потери времени, на поиск номера лот...",[шайдуллин р. ф.],[окулов а. м.]
1,2023-01-12,радуга,1,3,в оповещении об окончании срока действия серти...,добавить в оповещение информацию о реестре и о...,сокращение времени на выяснение какую сертифик...,[кайтаева в. в.],[окулов а. м.]
2,2023-01-12,NaN,1,3,при расчете ндс по бизнес-единицам: ао ппмтс п...,"в документе: ""псс:заявка на отгрузку"" реализов...",1.автоматизация учета\n2.точный расчет ндс\n3....,[хмелева н. в.],"[окулов а. м., other]"
3,2023-01-17,NaN,1,1,имеем базу данных кайдзен предложений.\nзатруд...,возможность фильтрации не по одному инициатору...,"получим ускоренную обратную связь, проверить с...",[лучников в. а.],[окулов а. м.]
4,2023-01-18,офис,1,2,отуствие уведомление о поступлении на склад ож...,сделать возможность получения уведомдения на п...,оперативное отслеживание поступления важных и ...,[чумаков а. в.],[окулов а. м.]
...,...,...,...,...,...,...,...,...,...
1122,2026-03-12,NaN,3,3,"для покраски коффердама грунтом, некоторые час...",металлические пластины для закрытия таких мест,процесс подготовки ускорится в несколько раз,"[авдаев м. м., гагарин д. н.]",[пьянков о. н.]
1123,2026-03-12,NaN,3,3,"для покраски коффердама грунтом, некоторые час...",вырезать кольца из магнитного листа,процесс подготовки ускорится в несколько раз,"[авдаев м. м., гагарин д. н.]",[пьянков о. н.]
1124,2026-03-12,NaN,3,3,"маляру, который пришел на другой участок, прих...",создать базу в электронном виде. где будет под...,другие маляры не будут отрываться от своей раб...,[авдаев м. м.],"[пьянков о. н., францкевич а. р.]"
1125,2026-03-12,NaN,3,3,"при покраске единичных, маленьких деталей. при...",приобрести малогабаритную камеру/печь.,быстрый процесс покраски мелких деталей. не пр...,[авдаев м. м.],[пьянков о. н.]


# 3. Подготовка target

In [21]:
mlb_exec = MultiLabelBinarizer()
y_exec = mlb_exec.fit_transform(df['executors'])

# список всех классов (исполнителей)
executor_classes = mlb_exec.classes_

# Проверка
print("Классы исполнителей:", executor_classes)
print("Форма таргета:", y_exec.shape)

Классы исполнителей: ['other' 'азовских н. а.' 'андреев в. а.' 'балтачев э. м.'
 'виниченко н. с.' 'гуляев а. а.' 'гусельников д. л.' 'дубовцева е. а.'
 'ежова ю. с.' 'кириллова к. с.' 'крамор о. а.' 'ледащев а. а.'
 'маркелов п. а.' 'мартюшев п. к.' 'огородникова т. а.' 'окулов а. м.'
 'политов м. п.' 'пчелякова в. а.' 'пьянков о. н.' 'редекоп а. г.'
 'рябинин а. в.' 'сергеева н. а.' 'соборная е. в.' 'тебеньков в. в.'
 'федотов е. а.' 'фоминых д. в.' 'францкевич а. р.' 'фурина с. н.'
 'хадиева г. р.' 'челноков а. а.' 'черанев в. н.' 'черанев ю. в.'
 'щеколов д. в.']
Форма таргета: (1127, 33)


In [22]:
counts = y_exec.sum(axis=0)
print(dict(zip(executor_classes, counts)))

{'other': np.int64(169), 'азовских н. а.': np.int64(13), 'андреев в. а.': np.int64(39), 'балтачев э. м.': np.int64(25), 'виниченко н. с.': np.int64(79), 'гуляев а. а.': np.int64(101), 'гусельников д. л.': np.int64(7), 'дубовцева е. а.': np.int64(8), 'ежова ю. с.': np.int64(23), 'кириллова к. с.': np.int64(55), 'крамор о. а.': np.int64(10), 'ледащев а. а.': np.int64(9), 'маркелов п. а.': np.int64(7), 'мартюшев п. к.': np.int64(47), 'огородникова т. а.': np.int64(50), 'окулов а. м.': np.int64(328), 'политов м. п.': np.int64(30), 'пчелякова в. а.': np.int64(9), 'пьянков о. н.': np.int64(68), 'редекоп а. г.': np.int64(21), 'рябинин а. в.': np.int64(10), 'сергеева н. а.': np.int64(127), 'соборная е. в.': np.int64(72), 'тебеньков в. в.': np.int64(69), 'федотов е. а.': np.int64(19), 'фоминых д. в.': np.int64(26), 'францкевич а. р.': np.int64(12), 'фурина с. н.': np.int64(10), 'хадиева г. р.': np.int64(9), 'челноков а. а.': np.int64(10), 'черанев в. н.': np.int64(64), 'черанев ю. в.': np.int64

# 4. Обучение

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier

from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

In [24]:
# Три отдельных TF-IDF вектора
# tfidf_have = TfidfVectorizer(max_features=2000, ngram_range=(1,2))
# X_have = tfidf_have.fit_transform(df['have'])

# tfidf_propose = TfidfVectorizer(max_features=2000, ngram_range=(1,2))
# X_propose = tfidf_propose.fit_transform(df['propose'])

# tfidf_benefit = TfidfVectorizer(max_features=2000, ngram_range=(1,2))
# X_benefit = tfidf_benefit.fit_transform(df['benefit'])

# Объединяем в одну матрицу признаков
# X_text = hstack([X_have, X_propose, X_benefit])

In [25]:
# X = hstack([X_text, X_init])  # X_init — multi-hot инициаторы

In [26]:
# y — multi-hot исполнители
# X_train, X_test, y_train, y_test = train_test_split(X, y_exec, test_size=0.2, random_state=42)

In [27]:
# # OneVsRestClassifier позволяет работать с multi-label
# clf = OneVsRestClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
# clf.fit(X_train, y_train)

# # Предсказания
# y_pred = clf.predict(X_test)

# 5. Оценка

### TF-IDF

In [28]:
# f1 = f1_score(y_test, y_pred, average='micro')
# print("Micro F1-score:", f1)

In [29]:
# print(classification_report(y_test, y_pred, target_names=mlb_exec.classes_))

          Без удаления маловстречающихся исполнителей
          micro avg       0.86      0.12      0.21       324
          macro avg       0.02      0.01      0.01       324
       weighted avg       0.27      0.12      0.15       324
        samples avg       0.17      0.14      0.15       324

        Появление Other исполнителей
         micro avg       0.86      0.12      0.21       322
         macro avg       0.03      0.01      0.01       322
      weighted avg       0.27      0.12      0.15       322
       samples avg       0.17      0.14      0.15       322

       Other на менее 10 записей
         micro avg       0.88      0.12      0.21       315
         macro avg       0.07      0.02      0.03       315
      weighted avg       0.28      0.12      0.15       315
       samples avg       0.17      0.14      0.15       315

# BERT

In [30]:
pip install transformers torch scikit-learn

In [31]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
# Исполнители — multi-label
mlb_exec = MultiLabelBinarizer()
y_exec = mlb_exec.fit_transform(df['executors'])

# Список классов
executor_classes = mlb_exec.classes_
num_classes = len(executor_classes)

In [33]:
# df['full_text'] = df['have'] + ' ' + df['propose'] + ' ' + df['benefit']

df['full_text'] = df.apply(
    lambda row: "инициаторы: " + ", ".join(row['initiators']) + " " +
                row['have'] + " " + row['propose'] + " " + row['benefit'],
    axis=1
)
texts = df['full_text'].tolist()

In [34]:
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [35]:
class KaizenDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.float)

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': label
        }

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, y_exec, test_size=0.2, random_state=42
)

train_dataset = KaizenDataset(X_train, y_train, tokenizer)
test_dataset = KaizenDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [37]:
class BertForMultiLabel(nn.Module):
    def __init__(self, num_classes):
        super(BertForMultiLabel, self).__init__()
        self.bert = AutoModel.from_pretrained("DeepPavlov/rubert-base-cased")
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # [CLS] токен
        x = self.dropout(pooled_output)
        return torch.sigmoid(self.out(x))  # multi-label

In [38]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertForMultiLabel(num_classes).to(device)

criterion = nn.BCELoss()  # бинарная кросс-энтропия для multi-label
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

epochs = 7

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1, Loss: 0.2915244766494684
Epoch 2, Loss: 0.1930721483209677
Epoch 3, Loss: 0.16975573149689457
Epoch 4, Loss: 0.15310362295100563
Epoch 5, Loss: 0.14115277664703235
Epoch 6, Loss: 0.1300815445812125
Epoch 7, Loss: 0.11909349667921401


In [39]:
model.eval()
all_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask)
        preds = (outputs > 0.3).int().cpu().numpy()
        all_preds.append(preds)

import numpy as np
y_pred = np.vstack(all_preds)

In [40]:
from sklearn.metrics import f1_score

f1 = f1_score(y_test, y_pred, average='micro')
print("Micro F1-score:", f1)

Micro F1-score: 0.4594594594594595


In [41]:
print("y_train sum:", y_train.sum(axis=0))  # сколько раз встречается каждая метка
print("y_train shape:", y_train.shape)
print("y_test sum:", y_test.sum(axis=0))

y_train sum: [137  12  31  19  61  75   4   5  21  42   6   6   5  36  40 261  27   8
  55  18   7  99  56  58  17  20  10  10   8  10  51  20  20]
y_train shape: (901, 33)
y_test sum: [32  1  8  6 18 26  3  3  2 13  4  3  2 11 10 67  3  1 13  3  3 28 16 11
  2  6  2  0  1  0 13  4  5]


# 6. Сохранение модели

In [42]:
# torch.save(model.state_dict(), 'bert_model.pt')

In [43]:
# tokenizer.save_pretrained('tokenizer/')

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

In [44]:
# import pickle

# with open('mlb_exec.pkl', 'wb') as f:
#     pickle.dump(mlb_exec, f)

In [45]:
# config = {
#     "threshold": 0.3,
#     "max_len": 128,
#     "num_classes": len(mlb_exec.classes_)
# }

# with open('config.pkl', 'wb') as f:
#     pickle.dump(config, f)

# Дубликаты

In [52]:
def get_embedding(text, model, tokenizer, device):
    model.eval()

    encoding = tokenizer(
        text,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        emb = outputs.last_hidden_state[:, 0]

    return emb.cpu().numpy()[0]

In [53]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

BertForMultiLabel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

In [54]:
def predict_top_k(text, model, tokenizer, mlb, k=3):
    model.eval()

    encoding = tokenizer(
        text,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(
            encoding['input_ids'],
            encoding['attention_mask']
        )
        probs = torch.sigmoid(outputs).cpu().numpy()[0]

    top_idx = probs.argsort()[-k:][::-1]

    return [(mlb.classes_[i], float(probs[i])) for i in top_idx]

In [57]:
def find_similar(text, embeddings, df, top_k=5):
    query_emb = get_embedding(text, model, tokenizer)

    sims = cosine_similarity([query_emb], embeddings)[0]
    top_idx = sims.argsort()[-top_k:][::-1]

    return df.iloc[top_idx][['full_text']]

In [59]:
df['embedding'] = df['full_text'].apply(
    lambda x: get_embedding(x, model, tokenizer, device)
)

embeddings = np.vstack(df['embedding'].values)

In [60]:
def predict_all(text):
    # 1. Исполнители
    executors = predict_top_k(text, model, tokenizer, mlb_exec, k=3)

    # 2. Похожие предложения
    similar = find_similar(text, embeddings, df, top_k=5)

    return {
        "executors": executors,
        "similar_cases": similar
    }

# Сохранение

In [65]:
import os

save_path = f"/content/drive/MyDrive/Hakaton/model/model_{f1:0.2}"
os.makedirs(save_path, exist_ok=True)

In [66]:
# модель
torch.save(model.state_dict(), f"{save_path}/bert_model.pt")

# токенайзер
tokenizer.save_pretrained(f"{save_path}/tokenizer")

# mlb
with open(f"{save_path}/mlb_exec.pkl", "wb") as f:
    pickle.dump(mlb_exec, f)

# dataframe
df.to_pickle(f"{save_path}/df.pkl")

# embeddings
np.save(f"{save_path}/embeddings.npy", embeddings)